# 15 · Forecasting avanzado: descomposición, ARIMA, ML y backtesting

Este notebook profundiza el lab 06. El foco es predecir futuro de forma honesta y comparar familias clásicas y Machine Learning.

## Objetivos
- Entender autocorrelación, estacionariedad y diferenciación.
- Usar seasonal decomposition y ACF/PACF.
- Entrenar ARIMA/SARIMA.
- Crear forecasting con gradient boosting y lags.
- Hacer backtesting rolling-origin.
- Tratar intervalos de predicción y múltiples horizontes.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
SEED=42
rng=np.random.default_rng(SEED); n=900; t=np.arange(n)
y=100+.03*t+8*np.sin(2*np.pi*t/7)+15*np.sin(2*np.pi*t/365)+rng.normal(0,3,n)
s=pd.Series(y,index=pd.date_range('2023-01-01',periods=n,freq='D'),name='y')
s.plot(figsize=(14,4)); plt.show()

## 1. Descomposición
Una representación clásica es $y_t=T_t+S_t+R_t$ (aditiva) o $y_t=T_tS_tR_t$ (multiplicativa). La descomposición ayuda a entender, pero no es un algoritmo universal de forecasting.


In [ ]:
seasonal_decompose(s,model='additive',period=7).plot(); plt.show()

## 2. Estacionariedad
ARIMA se formula de forma más natural sobre procesos estacionarios. Diferenciar puede eliminar tendencia. ADF prueba la hipótesis nula de raíz unitaria, pero no debe usarse mecánicamente: inspecciona datos y dominio.


In [ ]:
for label,x in [('original',s),('diff1',s.diff().dropna())]:
 stat,p,*_=adfuller(x); print(label,'ADF=',round(stat,3),'p=',p)
fig,ax=plt.subplots(1,2,figsize=(12,4)); plot_acf(s.diff().dropna(),lags=35,ax=ax[0]); plot_pacf(s.diff().dropna(),lags=35,ax=ax[1]); plt.show()

## 3. ARIMA / SARIMA
ARIMA(p,d,q) combina términos autoregresivos, diferenciación y media móvil. SARIMA agrega órdenes estacionales $(P,D,Q)_m$. Seleccionar órdenes solo mirando ACF/PACF es una guía; AIC/BIC, backtesting y parsimonia importan.


In [ ]:
train,test=s.iloc[:-90],s.iloc[-90:]
mod=SARIMAX(train,order=(2,1,2),seasonal_order=(1,0,1,7),enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
fc=mod.get_forecast(steps=len(test)); pred=fc.predicted_mean; ci=fc.conf_int()
print('SARIMA MAE',mean_absolute_error(test,pred))
plt.figure(figsize=(14,4)); plt.plot(test,label='real'); plt.plot(pred,label='forecast'); plt.fill_between(ci.index,ci.iloc[:,0],ci.iloc[:,1],alpha=.2); plt.legend(); plt.show()

## 4. Machine Learning con lags
Los árboles/boosting permiten incluir calendario, promociones, clima u otras covariables, además de lags. El riesgo es que deben estar disponibles para el horizonte futuro.


In [ ]:
def make_features(series):
 d=pd.DataFrame({'y':series})
 for lag in [1,2,3,7,14,28,365]: d[f'lag{lag}']=d.y.shift(lag)
 for w in [7,28]: d[f'mean{w}']=d.y.shift(1).rolling(w).mean(); d[f'std{w}']=d.y.shift(1).rolling(w).std()
 d['dow_sin']=np.sin(2*np.pi*d.index.dayofweek/7); d['dow_cos']=np.cos(2*np.pi*d.index.dayofweek/7)
 d['doy_sin']=np.sin(2*np.pi*d.index.dayofyear/365.25); d['doy_cos']=np.cos(2*np.pi*d.index.dayofyear/365.25)
 return d.dropna()
f=make_features(s); tr,te=f.iloc[:-90],f.iloc[-90:]; cols=f.columns.drop('y')
gb=HistGradientBoostingRegressor(max_iter=400,learning_rate=.05,max_leaf_nodes=20,l2_regularization=1,random_state=SEED).fit(tr[cols],tr.y)
p=gb.predict(te[cols]); print('GB MAE',mean_absolute_error(te.y,p)); plt.plot(te.index,te.y,label='real'); plt.plot(te.index,p,label='GB'); plt.legend(); plt.show()

## 5. Backtesting rolling-origin
Una sola ventana puede favorecer accidentalmente un modelo. Backtesting reentrena/actualiza a lo largo de varios cortes y simula cómo habría funcionado históricamente.


In [ ]:
def rolling_backtest(frame,features,horizon=30,min_train=400,step=30):
 rows=[]
 for end in range(min_train,len(frame)-horizon+1,step):
  tr=frame.iloc[:end]; va=frame.iloc[end:end+horizon]
  m=HistGradientBoostingRegressor(max_iter=250,learning_rate=.06,max_leaf_nodes=15,random_state=SEED).fit(tr[features],tr.y)
  p=m.predict(va[features]); rows.append([va.index[0],mean_absolute_error(va.y,p),mean_squared_error(va.y,p)**.5])
 return pd.DataFrame(rows,columns=['cutoff','MAE','RMSE'])
bt=rolling_backtest(f,cols); display(bt); print(bt[['MAE','RMSE']].mean())

## 6. Multi-horizon
- **Recursive:** predice t+1 y usa esa predicción para t+2; acumula errores.
- **Direct:** un modelo por horizonte.
- **Multi-output:** predice todos los horizontes conjuntamente.
- **Sequence models:** RNN/TCN/transformers procesan secuencias directamente.

## 7. Intervalos de predicción
Una predicción puntual sin incertidumbre es incompleta. SARIMA da intervalos paramétricos; en ML se pueden usar quantile regression, conformal prediction, bootstrap o ensembles.

## Temas para profundizar
ETS/Holt-Winters, TBATS, Prophet, XGBoost/LightGBM, N-BEATS, DeepAR, Temporal Fusion Transformer, PatchTST, conformal forecasting y hierarchical forecasting.

## Errores comunes
- shuffle de series;
- features que miran futuro;
- usar MAPE con ceros;
- no comparar contra seasonal-naive;
- ignorar cambio de régimen;
- usar covariables que no estarán disponibles en predicción.

## Ejercicios
1. Implementa Holt-Winters y compáralo.
2. Optimiza SARIMA por AIC y luego valida por backtesting.
3. Crea forecasting directo para horizontes 1, 7 y 30.
4. Añade quantile loss con LightGBM.
5. Simula una intervención y analiza cambio de régimen.
6. Implementa un prediction interval por conformal residuals.
